# 03 · Part B: the lever, how much it's used and what it costs

Part A found who keeps coming back to hospital. This notebook describes the tool that could change that: GLP-1 and SGLT2 drugs, which cut hospitalizations in large trials and are now recommended for diabetic patients with heart or kidney disease.

1. **How has their use changed?** CMS Part D national totals, 2013–2024 (same file every year).
2. **What does a year of treatment cost?** Gross Medicare cost per 30-day fill (CMS, 2024) vs. the Medicare negotiated prices (CMS fact sheets, `reference/assumptions.csv`).

**What this notebook hands to Part C (notebook 04):** the annual cost of treatment under each price.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # make src/ importable

import pandas as pd
from src.config import ROOT, PROCESSED, update_results
nat = pd.read_parquet(PROCESSED / "national_by_year.parquet")
A = pd.read_csv(ROOT / "reference" / "assumptions.csv").set_index("parameter")
A[["base", "unit", "source"]]

,base,unit,source
parameter,,,
rrr_hospitalization,0.110,relative reduction in all-cause hospitalization,EMPA-REG OUTCOME (empagliflozin vs placebo): a...
cost_per_admission_hcup_2018,15200.000,USD per readmission in 2018 dollars (all payers),AHRQ HCUP Statistical Brief #278: average 30-d...
price_sglt2_mfp_30d,197.000,USD per 30-day supply (2026),Jardiance Medicare negotiated price effective ...
price_sglt2b_mfp_30d,178.500,USD per 30-day supply (2026),Farxiga Medicare negotiated price effective Ja...
price_glp1_mfp_30d,274.000,USD per 30-day supply (2027),Semaglutide Medicare negotiated price effectiv...
benchmark_readmission_diabetes,0.182,30-day readmission rate of hospitalized diabet...,"Rubin & Shah, Current Diabetes Reports 2021: m..."
fda_first_glp1,2005.000,year of first FDA approval,"Exenatide (Byetta), first approved 2005-04-28 ..."
fda_first_sglt2,2013.000,year of first FDA approval,"Canagliflozin (Invokana), approved 2013-03-29 ..."


## 1. Use over time: 1.5% of claims in 2013, a third in 2024

Same CMS file every year, same definitions. "Newer" = any product containing a GLP-1 or SGLT2 drug (combination pills included). Weight-loss brands (Wegovy, Zepbound, Saxenda) are excluded.

In [2]:
trend = nat.groupby("year").apply(lambda d: pd.Series({
    "claims_M": d.Tot_Clms.sum() / 1e6, "cost_B": d.Tot_Drug_Cst.sum() / 1e9,
    "newer_share_claims": 100 * d.loc[d.newer, "Tot_Clms"].sum() / d.Tot_Clms.sum(),
    "newer_share_cost": 100 * d.loc[d.newer, "Tot_Drug_Cst"].sum() / d.Tot_Drug_Cst.sum()}), include_groups=False).round(1)
trend

,claims_M,cost_B,newer_share_claims,newer_share_cost
year,,,,
2013,75.1,10.4,1.5,5.4
2014,80.8,14.0,2.1,6.7
2015,84.2,17.4,3.2,9.3
2016,87.6,20.2,4.3,12.5
2017,89.2,23.1,5.5,16.5
2018,89.5,26.6,7.0,20.9
2019,90.9,30.5,9.2,26.6
2020,92.8,34.3,11.7,33.6
2021,94.5,39.4,15.2,41.5


In the hospital data (1999–2008) none of these patients were on either class: the first GLP-1 (exenatide, Byetta) was approved in 2005 and the first SGLT2 (canagliflozin, Invokana) in 2013 (FDA approval dates in `reference/assumptions.csv`). By 2024 they're **76% of Medicare diabetes drug spending but only 33% of claims**, because they cost far more per prescription. That price gap is the whole question for Part C.

## 2. What a year of treatment costs

**Gross cost** = what Medicare Part D paid per 30-day fill in 2024 (total drug cost ÷ 30-day fills, CMS national file), times 12. It's before manufacturer rebates, which are confidential.

**Negotiated price** = the price Medicare negotiated under the Inflation Reduction Act: Jardiance $197 and Farxiga $178.50 a month from 2026, semaglutide (Ozempic, Rybelsus) $274 a month from 2027. Sources are in `reference/assumptions.csv`.

In [3]:
y24 = nat[nat.year == 2024]
BRANDS = {"Jardiance": "SGLT2", "Farxiga": "SGLT2", "Ozempic": "GLP-1", "Trulicity": "GLP-1", "Mounjaro": "GLP-1", "Rybelsus": "GLP-1"}
NEGOTIATED = {"Jardiance": A.loc["price_sglt2_mfp_30d", "base"], "Farxiga": A.loc["price_sglt2b_mfp_30d", "base"],
              "Ozempic": A.loc["price_glp1_mfp_30d", "base"], "Rybelsus": A.loc["price_glp1_mfp_30d", "base"]}
cost = (y24[y24.Brnd_Name.isin(BRANDS)].groupby("Brnd_Name")
          .agg(patients=("Tot_Benes", "sum"), cost=("Tot_Drug_Cst", "sum"), fills=("Tot_30day_Fills", "sum"))
          .assign(cls=lambda d: d.index.map(BRANDS), gross_30d=lambda d: d.cost / d.fills)
          .assign(gross_year=lambda d: 12 * d.gross_30d, negotiated_30d=lambda d: d.index.map(NEGOTIATED).astype(float))
          .assign(negotiated_year=lambda d: 12 * d.negotiated_30d)
          .sort_values("cost", ascending=False))
cost[["cls", "patients", "gross_30d", "gross_year", "negotiated_30d", "negotiated_year"]].round(0)

,cls,patients,gross_30d,gross_year,negotiated_30d,negotiated_year
Brnd_Name,,,,,,
Ozempic,GLP-1,1951892.0,903.0,10840.0,274.0,3288.0
Jardiance,SGLT2,2542517.0,541.0,6491.0,197.0,2364.0
Mounjaro,GLP-1,894243.0,1040.0,12475.0,NaN,NaN
Trulicity,GLP-1,788414.0,948.0,11381.0,NaN,NaN
Farxiga,SGLT2,1268791.0,538.0,6460.0,178.0,2142.0
Rybelsus,GLP-1,309850.0,916.0,10989.0,274.0,3288.0


In [4]:
def gross_year(cls):
    c = cost[cost.cls == cls]
    return 12 * c.cost.sum() / c.fills.sum()

ANNUAL = {
    "SGLT2 · negotiated (2026)": 12 * A.loc["price_sglt2_mfp_30d", "base"],
    "GLP-1 · negotiated (2027)": 12 * A.loc["price_glp1_mfp_30d", "base"],
    "SGLT2 · gross 2024": cost.loc["Jardiance", "gross_year"],     # same drug as the negotiated price: Jardiance vs Jardiance
}
pd.Series(ANNUAL).round(0).rename("annual cost per patient ($)")

SGLT2 · negotiated (2026)    2364.0
GLP-1 · negotiated (2027)    3288.0
SGLT2 · gross 2024           6491.0
Name: annual cost per patient ($), dtype: float64

A year of Jardiance costs Medicare about **$6,500 at 2024 gross cost** and **$2,364 at the 2026 negotiated price**; both figures are for the same drug, so they compare like with like. A GLP-1 (semaglutide) costs about **$3,288 a year** at its 2027 negotiated price. These are the price scenarios in Part C (notebook 04). Gross cost is before confidential manufacturer rebates, so the true net price sits between the two.

In [5]:
update_results("lever", {
    "trend": trend.reset_index().to_dict("records"),
    "brand_costs": cost.reset_index()[["Brnd_Name", "cls", "patients", "gross_30d", "gross_year", "negotiated_30d", "negotiated_year"]].round(0).to_dict("records"),
    "annual_cost": {k: round(v) for k, v in ANNUAL.items()},
})
print("saved")

saved
